In [ ]:
! pip3 install -r requirements.txt

In [ ]:
!nvidia-smi

In [ ]:
import os
from utils.utils import get_offers
from utils.secrets_utils import get_secret
from utils.constants import GCP_PROJECT_ID,ENV_SHORT_NAME,BIGQUERY_ANALYTICS_DATASET
import vertexai
from google.cloud import aiplatform
import json
from vertexai.language_models import ChatModel, InputOutputTextPair


In [ ]:
offer_list = get_offers(number_offers=500,analytics_dataset="analytics_stg")

In [ ]:
# Cloud project id.
PROJECT_ID = GCP_PROJECT_ID,ENV_SHORT_NAME # @param {type:"string"}

# The region you want to launch jobs in.
# Select region based on the accelerators and regions supported by Vertex AI Prediction
# https://cloud.google.com/vertex-ai/docs/predictions/configure-compute.
REGION = "us-central1"  # @param {type:"string"}

# The Cloud Storage bucket for storing experiments output.
# Start with gs:// prefix, e.g. gs://foo_bucket.
BUCKET_URI = "gs://data-bucket-stg"  # @param {type:"string"}

! gcloud config set project $GCP_PROJECT_ID


STAGING_BUCKET = os.path.join(BUCKET_URI, "temporal")

# The service account looks like:
# '@.iam.gserviceaccount.com'
# Please go to https://cloud.google.com/iam/docs/service-accounts-create#iam-service-accounts-create-console
# and create service account with `Vertex AI User` and `Storage Object Admin` roles.
# The service account for deploying fine tuned model.
SERVICE_ACCOUNT = "cloud-run-dev@passculture-data-ehp.iam.gserviceaccount.com"  # @param {type:"string"}

In [ ]:
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)

In [ ]:
%%time
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda"  # the device to load the model onto
model_name = "mistralai/Mistral-7B-v0.1" # "mistralai/Mistral-7B-Instruct-v0.1" -> specialized in english and code
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", return_dict=True, torch_dtype=torch.float16
)





In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline("text-generation", model=model, tokenizer=tokenizer)


In [ ]:


prompt = "My favourite condiment is"

sequences = pipeline(
    prompt,
    max_length=200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for seq in sequences:
    print(f"Result: {seq['generated_text']}")

### https://huggingface.co/blog/accelerate-large-models